In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize']= (12,6)

In [ ]:
train_id = pd.read_csv('../../data/train_identity.csv')
test_id = pd.read_csv('../../data/test_identity.csv')

train_tr = pd.read_csv('../../data/train_transaction.csv')
test_tr = pd.read_csv('../../data/test_transaction.csv')

In [ ]:
df = pd.merge(train_tr, train_id, on='TransactionID', how='left')
print(f'Train shape: {df.shape}')

In [ ]:
print(df.columns)

In [ ]:
df.head()

## RAM Kullanımı ve Optimizasyon Analizi

In [ ]:
def analyze_memory_usage(df):
    memory_usage = df.memory_usage(deep=True)
    memory_usage_mb = memory_usage / 1024**2
    total_memory = memory_usage_mb.sum()
    dtype_memory = {}
    for col in df.columns:
        dtype = str(df[col].dtype)
        mem = memory_usage_mb[col]
        if dtype not in dtype_memory:
            dtype_memory[dtype] = {'memory': 0, 'columns': []}
        dtype_memory[dtype]['memory'] += mem
        dtype_memory[dtype]['columns'].append((col, mem))
    print("="*80)
    print(f"TOPLAM BELLEK KULLANIMI: {total_memory:.2f} MB")
    print("="*80)
    return dtype_memory

dtype_memory, all_cols_memory = analyze_memory_usage(df)

## Float Kolonlarının Optimizasyonu

In [ ]:
# Float kolonlarını int'e çevir
int_like_cols = []
for col in df.select_dtypes(include='float64').columns:
    if df[col].dropna().apply(lambda x: x == int(x)).all():
        int_like_cols.append(col)

conversion_summary = {'int8': [], 'int16': [], 'int32': []}
for col in int_like_cols:
    col_min = df[col].min()
    col_max = df[col].max()
    if pd.notna(col_min) and pd.notna(col_max):
        if col_min >= -127 and col_max <= 127:
            df[col] = pd.array(df[col], dtype='Int8')
            conversion_summary['int8'].append(col)
        elif col_min >= -32000 and col_max <= 32000:
            df[col] = pd.array(df[col], dtype='Int16')
            conversion_summary['int16'].append(col)
        else:
            df[col] = pd.array(df[col], dtype='Int32')
            conversion_summary['int32'].append(col)

print(f"int8: {len(conversion_summary['int8'])} kolonlar")
print(f"int16: {len(conversion_summary['int16'])} kolonlar")
print(f"int32: {len(conversion_summary['int32'])} kolonlar")

## Object Kolonlarının Category'ye Çevrilmesi

In [ ]:
unique_threshold = 65
object_cols = df.select_dtypes(include='object').columns

for col in object_cols:
    if df[col].nunique() < unique_threshold:
        df[col] = df[col].astype('category')

print(f"Object → Category: {len([col for col in df.columns if df[col].dtype == 'category'])} kolonlar")

In [ ]:
# Pickle ile kaydet
import pickle

with open('../../data/train_optimized.pkl', 'wb') as f:
    pickle.dump(df, f)

print("="*80)
print("✓ Optimized dataframe saved: train_optimized.pkl")
print(f"  Shape: {df.shape}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("="*80)